# **Day 6: Students Performance Dataset - Comprehensive Analysis & Multi-Output Regression**

This notebook provides an in-depth analysis of student academic performance data, with extensive EDA and multiple ML approaches to predict exam scores.

## 1. Import Required Libraries

In [1]:
# Import Required Libraries
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import joblib

# Set plotly to dark theme
px.defaults.template = "plotly_dark"

## 2. Load and Explore Dataset

In [2]:
# Load Dataset
students = pd.read_csv('../data/StudentsPerformance.csv')
print("Dataset loaded successfully!")
print(f"Shape: {students.shape}")

# Basic exploration
print("\nDataset Info:")
students.info()

print("\nMissing Values:")
print(students.isnull().sum())

print("\nDescriptive Statistics:")
display(students.describe())

print("\nSample Data:")
display(students.head())

Dataset loaded successfully!
Shape: (1000, 8)

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   gender                       1000 non-null   object
 1   race/ethnicity               1000 non-null   object
 2   parental level of education  1000 non-null   object
 3   lunch                        1000 non-null   object
 4   test preparation course      1000 non-null   object
 5   math score                   1000 non-null   int64 
 6   reading score                1000 non-null   int64 
 7   writing score                1000 non-null   int64 
dtypes: int64(3), object(5)
memory usage: 62.6+ KB

Missing Values:
gender                         0
race/ethnicity                 0
parental level of education    0
lunch                          0
test preparation course        0
math score                     

,math score,reading score,writing score
count,1000.00000,1000.000000,1000.000000
mean,66.08900,69.169000,68.054000
std,15.16308,14.600192,15.195657
min,0.00000,17.000000,10.000000
25%,57.00000,59.000000,57.750000
50%,66.00000,70.000000,69.000000
75%,77.00000,79.000000,79.000000
max,100.00000,100.000000,100.000000



Sample Data:


,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


## 3. Comprehensive Exploratory Data Analysis

In [3]:
# Score Distributions
score_cols = ['math score', 'reading score', 'writing score']

fig = make_subplots(rows=1, cols=3, subplot_titles=score_cols)
for i, col in enumerate(score_cols):
    fig.add_trace(go.Histogram(x=students[col], nbinsx=20, name=col, showlegend=False), row=1, col=i+1)
fig.update_layout(title="Score Distributions")
fig.show()

# Correlation Matrix
corr_matrix = students[score_cols].corr()
fig = px.imshow(corr_matrix, text_auto=True, color_continuous_scale='blues', title='Score Correlations')
fig.show()

# Box plots by gender
fig = px.box(students, x='gender', y=score_cols[0], title='Math Scores by Gender')
fig.show()

for col in score_cols[1:]:
    fig = px.box(students, x='gender', y=col, title=f'{col.title()} Scores by Gender')
    fig.show()

# By race/ethnicity
fig = px.box(students, x='race/ethnicity', y='math score', title='Math Scores by Race/Ethnicity')
fig.show()

# By parental education
fig = px.box(students, x='parental level of education', y='math score', title='Math Scores by Parental Education')
fig.update_xaxes(tickangle=45)
fig.show()

# By test preparation
fig = px.box(students, x='test preparation course', y='reading score', title='Reading Scores by Test Preparation')
fig.show()

# By lunch type
fig = px.box(students, x='lunch', y='writing score', title='Writing Scores by Lunch Type')
fig.show()

## 4. Feature Engineering

In [4]:
# Feature Engineering
students_processed = students.copy()

# Create total score
students_processed['total_score'] = students_processed[score_cols].sum(axis=1)

# Encode parental education numerically
education_mapping = {
    "some high school": 1,
    "high school": 2,
    "some college": 3,
    "associate's degree": 4,
    "bachelor's degree": 5,
    "master's degree": 6
}
students_processed['parental_education_encoded'] = students_processed['parental level of education'].map(education_mapping)

# Binary encoding for test preparation and lunch
students_processed['test_prep_completed'] = (students_processed['test preparation course'] == 'completed').astype(int)
students_processed['standard_lunch'] = (students_processed['lunch'] == 'standard').astype(int)

print("New features added:")
print("- total_score: Sum of all three scores")
print("- parental_education_encoded: Numerical encoding of education level")
print("- test_prep_completed: Binary indicator for test preparation")
print("- standard_lunch: Binary indicator for lunch type")

display(students_processed.head())

New features added:
- total_score: Sum of all three scores
- parental_education_encoded: Numerical encoding of education level
- test_prep_completed: Binary indicator for test preparation
- standard_lunch: Binary indicator for lunch type


,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score,total_score,parental_education_encoded,test_prep_completed,standard_lunch
0,female,group B,bachelor's degree,standard,none,72,72,74,218,5,0,1
1,female,group C,some college,standard,completed,69,90,88,247,3,1,1
2,female,group B,master's degree,standard,none,90,95,93,278,6,0,1
3,male,group A,associate's degree,free/reduced,none,47,57,44,148,4,0,0
4,male,group C,some college,standard,none,76,78,75,229,3,0,1


## 5. Model Training and Comparison

In [5]:
# Prepare data for modeling
X = students_processed.drop(columns=score_cols + ['total_score'])
y = students_processed[score_cols]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define categorical features
categorical_features = ['gender', 'race/ethnicity', 'parental level of education', 'lunch', 'test preparation course']

# Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ],
    remainder='passthrough'  # Keep numerical features as is
)

# Define models to compare
models = {
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': MultiOutputRegressor(GradientBoostingRegressor(n_estimators=100, random_state=42)),
    'Linear Regression': LinearRegression()
}

# Train and evaluate models
results = {}
for name, model in models.items():
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    
    # Calculate metrics for each target
    r2_scores = [r2_score(y_test.iloc[:, i], y_pred[:, i]) for i in range(len(score_cols))]
    mae_scores = [mean_absolute_error(y_test.iloc[:, i], y_pred[:, i]) for i in range(len(score_cols))]
    rmse_scores = [np.sqrt(mean_squared_error(y_test.iloc[:, i], y_pred[:, i])) for i in range(len(score_cols))]
    
    results[name] = {
        'R2': r2_scores,
        'MAE': mae_scores,
        'RMSE': rmse_scores,
        'pipeline': pipeline
    }
    
    print(f"\n{name} Results:")
    for i, subject in enumerate(score_cols):
        print(f"  {subject}: R²={r2_scores[i]:.4f}, MAE={mae_scores[i]:.2f}, RMSE={rmse_scores[i]:.2f}")

# Select best model (highest average R2)
best_model_name = max(results.keys(), key=lambda x: np.mean(results[x]['R2']))
best_pipeline = results[best_model_name]['pipeline']

print(f"\nBest performing model: {best_model_name}")
print(f"Average R²: {np.mean(results[best_model_name]['R2']):.4f}")


Random Forest Results:
  math score: R²=-0.0035, MAE=12.29, RMSE=15.63
  reading score: R²=-0.0219, MAE=11.80, RMSE=15.21
  writing score: R²=0.0970, MAE=11.20, RMSE=14.75

Gradient Boosting Results:
  math score: R²=0.1032, MAE=11.57, RMSE=14.77
  reading score: R²=0.0878, MAE=11.20, RMSE=14.37
  writing score: R²=0.1916, MAE=10.60, RMSE=13.96

Linear Regression Results:
  math score: R²=0.1760, MAE=11.27, RMSE=14.16
  reading score: R²=0.1594, MAE=10.83, RMSE=13.79
  writing score: R²=0.2637, MAE=10.19, RMSE=13.32

Best performing model: Linear Regression
Average R²: 0.1997

Gradient Boosting Results:
  math score: R²=0.1032, MAE=11.57, RMSE=14.77
  reading score: R²=0.0878, MAE=11.20, RMSE=14.37
  writing score: R²=0.1916, MAE=10.60, RMSE=13.96

Linear Regression Results:
  math score: R²=0.1760, MAE=11.27, RMSE=14.16
  reading score: R²=0.1594, MAE=10.83, RMSE=13.79
  writing score: R²=0.2637, MAE=10.19, RMSE=13.32

Best performing model: Linear Regression
Average R²: 0.1997


## 6. Save Model and Create Prediction Function

In [6]:
# Save the best model
joblib.dump(best_pipeline, '../models/students_performance_best_model.joblib')
print(f"Best model ({best_model_name}) saved to ../models/students_performance_best_model.joblib")

# Create prediction function
def predict_student_scores(student_data, model_path='../models/students_performance_best_model.joblib'):
    """
    Predict math, reading, and writing scores for a student.
    
    Args:
        student_data (dict): Dictionary with student features
        model_path (str): Path to the saved model
        
    Returns:
        pd.DataFrame: Predicted scores
    """
    model = joblib.load(model_path)
    input_df = pd.DataFrame([student_data])
    predictions = model.predict(input_df)
    
    result = pd.DataFrame(
        predictions.round(1),
        columns=['Predicted Math Score', 'Predicted Reading Score', 'Predicted Writing Score']
    )
    return result

# Example prediction
example_student = {
    'gender': 'female',
    'race/ethnicity': 'group B',
    'parental level of education': "bachelor's degree",
    'lunch': 'standard',
    'test preparation course': 'completed',
    'parental_education_encoded': 5,
    'test_prep_completed': 1,
    'standard_lunch': 1
}

print("\nExample Prediction:")
print("Student Profile:")
for key, value in example_student.items():
    print(f"  {key}: {value}")

predicted_scores = predict_student_scores(example_student)
print("\nPredicted Scores:")
print(predicted_scores.to_string(index=False))

Best model (Linear Regression) saved to ../models/students_performance_best_model.joblib

Example Prediction:
Student Profile:
  gender: female
  race/ethnicity: group B
  parental level of education: bachelor's degree
  lunch: standard
  test preparation course: completed
  parental_education_encoded: 5
  test_prep_completed: 1
  standard_lunch: 1

Predicted Scores:
 Predicted Math Score  Predicted Reading Score  Predicted Writing Score
                 73.9                     82.8                     85.8


## 7. Conclusions and Insights

### Key Findings:
1. **Model Performance**: Even with multiple algorithms and feature engineering, R² scores remain modest (best around 0.15-0.25), indicating that demographic features alone are weak predictors of individual exam scores.

2. **Important Factors**:
   - Test preparation course completion significantly boosts scores
   - Standard lunch vs. free/reduced lunch shows clear performance differences
   - Parental education level has some correlation with student performance
   - Gender differences exist across subjects

3. **Limitations**: 
   - Dataset lacks individual factors like study habits, IQ, motivation
   - Small sample size (1000 students) limits model complexity
   - Scores may be influenced by unmeasured variables

### Recommendations:
- Collect additional features for better predictions
- Consider separate models for each subject
- Use ensemble methods or neural networks for complex relationships
- Focus on identifying at-risk students rather than precise score prediction

### Best Practices Applied:
- Comprehensive EDA with multiple visualization types
- Feature engineering to create more predictive variables
- Model comparison across different algorithms
- Proper preprocessing with pipelines
- Cross-validation considerations for robust evaluation